In [ ]:
from langgraph.graph import StateGraph
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from typing import TypedDict
from langgraph.constants import END, START

In [2]:
load_dotenv()

True

In [3]:
class LLMstate(TypedDict):

    question: str
    answer: str

In [4]:
llm = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True,
    )
)
model = ChatHuggingFace(llm=llm)

/Users/harshitjadiya/Desktop/LangGraph/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3396.29it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [10]:
def llm_qa(state: LLMstate) -> LLMstate:
    question = state['question']

    prompt = f"Answer the following question {question}"

    answer = model.invoke(prompt).content

    state['answer'] = answer

    return state

In [11]:
graph = StateGraph(LLMstate)

graph.add_node("llm_qa", llm_qa)
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

workflow = graph.compile()

In [12]:
final_output = workflow.invoke({'question': "What is distance between the sun and the moon?"})

In [16]:
print(final_output)

{'question': 'What is distance between the sun and the moon?', 'answer': '<|user|>\nAnswer the following question What is distance between the sun and the moon?</s>\n<|assistant|>\nThe distance between the sun and the moon is approximately 1.5 million miles (2.4 million kilometers).'}
